# Estudo de Caso 5.1 — Hidrodinâmica do Rio Macaé: Propagação de Onda de Cheia

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 5 — Hidrodinâmica de Canais Abertos |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 120 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Diferenças Finitas — Esquemas Explícito e Implícito |
| **Bibliografia** | Lugon Junior et al. (2026); Gobbi (s.d.), Cap. 2 e 5 |

---

## 📋 Resumo

Neste estudo de caso, simulamos a propagação de uma onda de cheia no Rio Macaé (RJ) utilizando o modelo de onda cinemática (simplificação das equações de Saint-Venant). Implementamos dois esquemas de diferenças finitas — **explícito** (upwind, condicionalmente estável) e **implícito** (incondicionalmente estável, com solução de sistema tridiagonal) — e comparamos seus resultados em termos de precisão, estabilidade e custo computacional.

O caso utiliza parâmetros reais do Rio Macaé (largura, declividade, rugosidade de Manning) e um hidrograma de cheia sintético com pico de 12,0 m³/s.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. Aplicar a equação de Manning para verificar a consistência dos parâmetros hidráulicos
2. Formular as equações de Saint-Venant no modelo de onda cinemática
3. Implementar a condição CFL e calcular o passo de tempo máximo para esquemas explícitos
4. Resolver numericamente a equação de continuidade com esquemas explícito e implícito
5. Montar e resolver sistemas tridiagonais com `scipy.linalg.solve_banded`
6. Comparar esquemas numéricos quanto à estabilidade, precisão e custo computacional
7. Analisar atenuação de pico, tempo de trânsito e dispersão numérica

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from scipy.linalg import solve_banded
from IPython.display import display, Markdown

# Configuração de estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🌊 Seção 1: Parâmetros do Rio Macaé

O Rio Macaé é um rio de planície com baixa declividade e vazão moderada. Os parâmetros hidráulicos foram obtidos de dados de campo e levantamentos batimétricos.

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:--------|:------|:--------|
| Largura média do canal | $B$ | 42,2 | m |
| Profundidade inicial | $h_0$ | 0,71 | m |
| Declividade do fundo | $S_0$ | 1,0 × 10⁻⁴ | m/m |
| Coeficiente de Manning | $n$ | 0,035 | s/m^(1/3) |
| Vazão de referência | $Q_{\text{ref}}$ | 6,7 | m³/s |
| Comprimento do trecho | $L$ | 6 300 | m |
| Aceleração da gravidade | $g$ | 9,81 | m/s² |

In [ ]:
# ============================================================
# PARÂMETROS DO RIO MACAÉ
# ============================================================

# Geometria do canal (seção retangular larga)
B = 42.2            # Largura média [m]
h0 = 0.71           # Profundidade inicial (regime uniforme) [m]
S0 = 1.0e-4         # Declividade do fundo [m/m]

# Propriedades hidráulicas
n_manning = 0.035   # Coeficiente de rugosidade de Manning [s/m^(1/3)]
Q_ref = 6.7         # Vazão de referência [m³/s]
g = 9.81            # Aceleração da gravidade [m/s²]

# Domínio computacional
L = 6300.0          # Comprimento do trecho [m]
dx = 100.0          # Espaçamento espacial [m]
N = int(L / dx)     # Número de intervalos espaciais
x = np.linspace(0, L, N + 1)  # Vetor de posições

# Passo de tempo
dt_exp = 30.0       # [s] — Esquema explícito (limitado pelo CFL)
dt_imp = 150.0      # [s] — Esquema implícito (pode ser maior)

# Tempo total de simulação
t_total = 8.0 * 3600.0  # [s] — 8 horas

print("=" * 60)
print(" PARÂMETROS DO RIO MACAÉ")
print("=" * 60)
print(f"\n📊 Geometria:")
print(f"  • Largura: B = {B} m")
print(f"  • Profundidade: h₀ = {h0} m")
print(f"  • Declividade: S₀ = {S0:.1e}")
print(f"\n🔧 Hidráulica:")
print(f"  • Manning: n = {n_manning} s/m^(1/3)")
print(f"  • Vazão ref.: Q_ref = {Q_ref} m³/s")
print(f"\n📐 Discretização:")
print(f"  • Comprimento: L = {L/1000:.1f} km")
print(f"  • Δx = {dx} m")
print(f"  • Número de intervalos: N = {N}")
print(f"  • Número de nós: N+1 = {N+1}")
print(f"\n⏱️ Passos de tempo:")
print(f"  • Explícito: Δt = {dt_exp} s")
print(f"  • Implícito: Δt = {dt_imp} s")

---

## 🔬 Seção 2: Verificação com Manning

Antes de simular, verificamos se os parâmetros são consistentes com a equação de Manning. Para seção retangular larga ($R_h \approx h$), a vazão é:

$$
Q_{\text{Manning}} = \frac{B}{n} \, h^{5/3} \, S_0^{1/2}
$$

Esta verificação garante que a condição inicial $h_0$ é coerente com $Q_{\text{ref}}$.

> **📖 Referência:** Gobbi (s.d.), Seção 5.1.1, Eq. 5.7; Lugon Junior et al. (2026), Cap. 5.

In [ ]:
# ============================================================
# VERIFICAÇÃO COM MANNING
# ============================================================

# Constante da equação de Manning para canal retangular largo (Rh ≈ h)
# Q = (B/n) * h^(5/3) * S0^(1/2)  =>  Q = alpha * h^(5/3)
alpha = (B / n_manning) * (S0 ** 0.5)

# Calcular vazão pela equação de Manning com h0
Q_Manning = alpha * h0 ** (5.0 / 3.0)

# Calcular profundidade normal para Q_ref
h_n = (Q_ref / alpha) ** (3.0 / 5.0)

print("=" * 60)
print(" VERIFICAÇÃO COM MANNING")
print("=" * 60)
print(f"\n🧮 Cálculo:")
print(f"  • Q_Manning = (B/n)·h₀^(5/3)·√S₀")
print(f"  • Q_Manning = ({B}/{n_manning})×{h0}^(5/3)×√{S0}")
print(f"  • Q_Manning = {Q_Manning:.3f} m³/s")
print(f"\n✅ Comparação:")
print(f"  • Q_ref (dado)          = {Q_ref} m³/s")
print(f"  • Q_Manning (calculado) = {Q_Manning:.3f} m³/s")
print(f"  • Erro relativo         = {abs(Q_Manning - Q_ref)/Q_ref * 100:.2f}%")
print(f"\n📏 Profundidade normal para Q_ref:")
print(f"  • h_n = (Q_ref / alpha)^(3/5) = {h_n:.4f} m")
print(f"  • h₀ (inicial) = {h0} m")
print(f"  • Diferença = {abs(h_n - h0) * 100:.1f} cm")
print(f"\n💡 Conclusão: Os parâmetros são consistentes (erro < 5%).")

---

## 📐 Seção 3: Formulação Matemática — Onda Cinemática

As equações de Saint-Venant completas para escoamento unidimensional em canais abertos são:

**Conservação da massa (continuidade):**
$$
\frac{\partial A}{\partial t} + \frac{\partial Q}{\partial x} = 0
$$

**Conservação da quantidade de movimento:**
$$
\frac{\partial Q}{\partial t} + \frac{\partial}{\partial x}\left(\frac{Q^2}{A}\right) + gA\frac{\partial h}{\partial x} = gA(S_0 - S_f)
$$

No modelo de **onda cinemática**, desprezam-se os termos de inércia e gradiente de pressão, resultando na relação algébrica de Manning:

$$
Q = \frac{B}{n} \, h^{5/3} \, S_0^{1/2} = \alpha \, h^{5/3}
$$

A equação governante fica:

$$
\frac{\partial h}{\partial t} + \frac{1}{B}\frac{\partial Q}{\partial x} = 0
$$

A **celeridade da onda cinemática** é:

$$
c_k = \frac{dQ}{dA} = \frac{5}{3} \frac{Q}{Bh} = \frac{5}{3} u
$$

> **📖 Referência:** Gobbi (s.d.), Seção 5.1.1, Eqs. 5.4–5.7.

### Condições de contorno

- **Montante ($x = 0$):** Hidrograma prescrito $Q(0, t)$ → $h(0,t) = (Q/\alpha)^{3/5}$
- **Jusante ($x = L$):** Condição de onda saindo (Neumann): $\partial h / \partial x = 0$

### Condição inicial

$$
h(x, 0) = h_0 \quad \text{(regime uniforme)}
$$

---

## ⚡ Seção 4: Condição CFL e Estabilidade

A condição de Courant-Friedrichs-Lewy (CFL) garante que a informação não "pule" mais de uma célula por passo de tempo. Para o modelo de onda cinemática:

$$
\mathrm{CFL} = \frac{c_k \, \Delta t}{\Delta x} \leq 1
$$

onde $c_k$ é a celeridade da onda cinemática. Para o **esquema explícito**, o passo de tempo deve satisfazer essa condição. Para o **esquema implícito**, não há restrição de estabilidade (incondicionalmente estável).

> **📖 Referência:** Lugon Junior et al. (2026), Cap. 5.

In [ ]:
# ============================================================
# FUNÇÕES AUXILIARES E CONDIÇÃO CFL
# ============================================================

def h_from_Q(Q):
    """Converte vazão em lâmina d'água usando a relação de Manning."""
    return (Q / alpha) ** (3.0 / 5.0)

def Q_from_h(h):
    """Converte lâmina d'água em vazão."""
    return alpha * np.asarray(h) ** (5.0 / 3.0)

def celeridade(h):
    """Celeridade da onda cinemática: ck = (5/3) * u."""
    h = np.asarray(h)
    u = Q_from_h(h) / (B * h)
    return (5.0 / 3.0) * u

# Cálculo do CFL para a condição inicial
u_inicial = Q_ref / (B * h0)
ck_inicial = celeridade(h0)
cfl_exp = (ck_inicial * dt_exp) / dx
cfl_imp = (ck_inicial * dt_imp) / dx
Fr = u_inicial / np.sqrt(g * h0)  # Número de Froude

print("=" * 60)
print(" CONDIÇÃO CFL E ESTABILIDADE")
print("=" * 60)
print(f"\n📊 Parâmetros iniciais:")
print(f"  • Velocidade média: u = {u_inicial:.4f} m/s")
print(f"  • Celeridade cinemática: ck = {ck_inicial:.4f} m/s")
print(f"  • Número de Froude: Fr = {Fr:.4f}")
print(f"\n🧮 Condição CFL:")
print(f"  • Esquema explícito (Δt = {dt_exp} s):")
print(f"    CFL = ck·Δt/Δx = {ck_inicial:.4f}×{dt_exp}/{dx} = {cfl_exp:.4f}")
print(f"    {'✅ Estável' if cfl_exp <= 1.0 else '❌ INSTÁVEL'} (CFL {'≤' if cfl_exp <= 1.0 else '>'} 1.0)")
print(f"\n  • Esquema implícito (Δt = {dt_imp} s):")
print(f"    CFL = ck·Δt/Δx = {ck_inicial:.4f}×{dt_imp}/{dx} = {cfl_imp:.4f}")
print(f"    ✅ Incondicionalmente estável (CFL > 1 permitido)")
print(f"\n💡 Interpretação:")
print(f"  • Fr = {Fr:.3f} < 1 → regime FLUVIAL (subcrítico)")
print(f"  • O modelo de onda cinemática é válido para Fr < 1")

---

## 🌪️ Seção 5: Hidrograma de Entrada

Simularemos um evento de cheia com:
- Vazão inicial: $Q = 6{,}7$ m³/s (regime permanente)
- Pico de cheia: $Q_{\max} = 12{,}0$ m³/s
- Rampa crescente: $t = 1{,}5$ h até $t = 2{,}0$ h
- Pico mantido: $t = 2{,}0$ h até $t = 6{,}0$ h
- Rampa decrescente: $t = 6{,}0$ h até $t = 6{,}5$ h

In [ ]:
# ============================================================
# HIDROGRAMA DE ENTRADA
# ============================================================

def hidrograma_entrada(t_horas):
    """
    Define o hidrograma de entrada Q(0, t).
    Pico de 12.0 m³/s entre 2h e 6h, com rampas de 1.5h a 2h e 6h a 6.5h.
    """
    if t_horas < 1.5:
        return Q_ref
    elif t_horas <= 2.0:
        return Q_ref + (12.0 - Q_ref) * (t_horas - 1.5) / 0.5
    elif t_horas <= 6.0:
        return 12.0
    elif t_horas <= 6.5:
        return 12.0 - (12.0 - Q_ref) * (t_horas - 6.0) / 0.5
    else:
        return Q_ref

# Plot do hidrograma de entrada
t_plot = np.linspace(0, 8 * 3600, 1000)
Q_plot = [hidrograma_entrada(t / 3600.0) for t in t_plot]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_plot / 3600, Q_plot, 'b-', linewidth=2)
ax.axhline(y=Q_ref, color='gray', linestyle='--', label=f'$Q_{{ref}}$ = {Q_ref} m³/s')
ax.axhline(y=12.0, color='red', linestyle='--', label='$Q_{{max}}$ = 12 m³/s')
ax.set_xlabel('Tempo [h]', fontsize=12)
ax.set_ylabel('Vazão [m³/s]', fontsize=12)
ax.set_title('Hidrograma de Entrada — Rio Macaé', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Hidrograma definido com sucesso!")

---

## 🧮 Seção 6: Simulação Numérica

### 6.1 Esquema Explícito (Upwind)

A discretização explícita upwind (para $u > 0$) da equação de continuidade é:

$$
h_i^{n+1} = h_i^n - \frac{\Delta t}{\Delta x \, B}\left(Q_i^n - Q_{i-1}^n\right)
$$

### 6.2 Esquema Implícito (Upwind)

A discretização implícita upwind resulta em:

$$
h_i^{n+1} + \frac{\Delta t}{\Delta x \, B}\left(Q_i^{n+1} - Q_{i-1}^{n+1}\right) = h_i^n
$$

Como $Q = \alpha h^{5/3}$ é não-linear em $h$, linearizamos:

$$
Q^{n+1} \approx Q^n + \left.\frac{dQ}{dh}\right|^n (h^{n+1} - h^n)
$$

onde $dQ/dh = c_k \cdot B$. Isso gera um sistema tridiagonal $\mathbf{A}\,\mathbf{h}^{n+1} = \mathbf{b}$, resolvido com `scipy.linalg.solve_banded`.

In [ ]:
# ============================================================
# 6.1 SIMULAÇÃO — ESQUEMA EXPLÍCITO
# ============================================================

def simular_explicito(dt):
    """
    Simula a propagação da onda de cheia com esquema explícito upwind.
    Código vetorizado (operações NumPy em arrays).
    """
    h = np.ones(N + 1) * h0  # Condição inicial: regime uniforme
    historia_h = [h.copy()]
    historia_t = [0.0]

    t = 0.0
    passo = 0

    while t < t_total:
        t_next = t + dt
        t_horas = t_next / 3600.0

        # 1. Calcular vazões em todos os nós
        Q = Q_from_h(h)

        # 2. Atualizar lâmina d'água (Upwind para u > 0)
        h_new = h.copy()
        h_new[1:] = h[1:] - (dt / (dx * B)) * (Q[1:] - Q[:-1])

        # 3. Condições de contorno
        h_new[0] = h_from_Q(hidrograma_entrada(t_horas))   # Montante
        h_new[-1] = h_new[-2]                               # Jusante (Neumann)

        h = h_new
        historia_h.append(h.copy())
        historia_t.append(t_next)
        t = t_next
        passo += 1

    return np.array(historia_t), np.array(historia_h), passo

print("Executando esquema explícito...")
t_inicio = time.time()
t_exp, h_exp, passos_exp = simular_explicito(dt_exp)
tempo_exp = time.time() - t_inicio
print(f"✅ Concluído em {tempo_exp:.4f} segundos ({passos_exp} passos de tempo).")

In [ ]:
# ============================================================
# 6.2 SIMULAÇÃO — ESQUEMA IMPLÍCITO
# ============================================================

def simular_implicito(dt):
    """
    Simula a propagação da onda de cheia com esquema implícito upwind.
    Monta e resolve sistema tridiagonal a cada passo de tempo.
    """
    h = np.ones(N + 1) * h0  # Condição inicial
    historia_h = [h.copy()]
    historia_t = [0.0]

    t = 0.0
    passo = 0

    while t < t_total:
        t_next = t + dt
        t_horas = t_next / 3600.0

        # Linearização: Q^{n+1} ≈ Q^n + (dQ/dh)^n * (h^{n+1} - h^n)
        Q_n = Q_from_h(h)
        ck = celeridade(h)
        dQ_dh = ck * B

        # Montagem do sistema tridiagonal A * h_new = b
        main_diag = np.ones(N + 1)
        sub_diag = np.zeros(N + 1)
        b_vec = h.copy()

        # Nó 0 (Contorno de montante prescrito)
        main_diag[0] = 1.0
        b_vec[0] = h_from_Q(hidrograma_entrada(t_horas))

        # Nós internos (1 a N-1)
        for i in range(1, N):
            coef_i = (dt / (dx * B)) * dQ_dh[i]
            coef_im1 = (dt / (dx * B)) * dQ_dh[i - 1]

            main_diag[i] = 1.0 + coef_i
            sub_diag[i] = -coef_im1

            b_vec[i] = h[i] + (dt / (dx * B)) * (
                dQ_dh[i] * h[i] - dQ_dh[i - 1] * h[i - 1]
                - Q_n[i] + Q_n[i - 1]
            )

        # Nó N (Contorno de jusante: h_N = h_{N-1})
        main_diag[N] = 1.0
        sub_diag[N] = -1.0
        b_vec[N] = 0.0

        # Montar matriz bandada para scipy
        ab = np.zeros((3, N + 1))
        ab[0, 1:] = 0.0          # Diagonal superior (vazia neste esquema upwind)
        ab[1, :] = main_diag     # Diagonal principal
        ab[2, :-1] = sub_diag[1:]  # Diagonal inferior

        # Resolver o sistema linear
        h_new = solve_banded((1, 1), ab, b_vec)

        historia_h.append(h_new.copy())
        historia_t.append(t_next)
        h = h_new
        t = t_next
        passo += 1

    return np.array(historia_t), np.array(historia_h), passo

print("Executando esquema implícito...")
t_inicio = time.time()
t_imp, h_imp, passos_imp = simular_implicito(dt_imp)
tempo_imp = time.time() - t_inicio
print(f"✅ Concluído em {tempo_imp:.4f} segundos ({passos_imp} passos de tempo).")

---

## 📊 Seção 7: Visualização dos Resultados

Plotamos os hidrogramas em três seções (montante, meio e jusante) e o perfil longitudinal da lâmina d'água durante o pico da cheia.

In [ ]:
# ============================================================
# VISUALIZAÇÃO: HIDROGRAMAS E PERFIL LONGITUDINAL
# ============================================================

# Encontrar o índice de tempo mais próximo de 3.0 horas (pico em trânsito)
idx_pico_exp = np.argmin(np.abs(t_exp / 3600.0 - 3.0))
idx_pico_imp = np.argmin(np.abs(t_imp / 3600.0 - 3.0))

# --- Figura 1: Hidrogramas em três seções ---
secoes = [0, N // 2, N]
nomes_secoes = [
    'Montante (x = 0 km)',
    f'Meio (x = {x[N//2]/1000:.1f} km)',
    f'Jusante (x = {L/1000:.1f} km)'
]

plt.figure(figsize=(14, 10))
for i, (idx_sec, nome) in enumerate(zip(secoes, nomes_secoes)):
    plt.subplot(3, 1, i + 1)

    # Interpolar para comparar no mesmo eixo de tempo
    t_comum = np.linspace(0, 8, 500)
    Q_exp_sec = np.interp(t_comum, t_exp / 3600.0, Q_from_h(h_exp[:, idx_sec]))
    Q_imp_sec = np.interp(t_comum, t_imp / 3600.0, Q_from_h(h_imp[:, idx_sec]))

    plt.plot(t_comum, Q_exp_sec, 'b-', label=f'Explícito (Δt={dt_exp:.0f}s)', linewidth=2)
    plt.plot(t_comum, Q_imp_sec, 'r--', label=f'Implícito (Δt={dt_imp:.0f}s)', linewidth=2)

    plt.title(f'Hidrograma na seção: {nome}')
    plt.ylabel('Vazão Q (m³/s)')
    plt.xlabel('Tempo (horas)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Figura 2: Perfil longitudinal no momento do pico ---
plt.figure(figsize=(10, 5))
plt.plot(x / 1000, h_exp[idx_pico_exp], 'b-',
         label=f'Explícito (t = {t_exp[idx_pico_exp]/3600:.1f} h)', linewidth=2)
plt.plot(x / 1000, h_imp[idx_pico_imp], 'r--',
         label=f'Implícito (t = {t_imp[idx_pico_imp]/3600:.1f} h)', linewidth=2)
plt.axhline(h0, color='gray', linestyle=':', label=f'Nível inicial ({h0} m)')

plt.title("Perfil Longitudinal da Lâmina d'Água durante o Pico da Cheia")
plt.xlabel('Distância longitudinal x (km)')
plt.ylabel("Lâmina d'água h (m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 Observação:")
print("  • A onda se propaga de montante (esquerda) para jusante (direita)")
print("  • O pico de vazão diminui ao longo do rio (atenuação)")
print("  • A frente da onda se alarga (dispersão numérica)")

---

## 📈 Seção 8: Análise Quantitativa e Tabela Comparativa

Calculamos métricas de desempenho e comparamos os dois esquemas.

In [ ]:
# ============================================================
# ANÁLISE QUANTITATIVA
# ============================================================

# Atenuação de pico na jusante
Q_max_saida_exp = np.max(Q_from_h(h_exp[:, N]))
Q_max_saida_imp = np.max(Q_from_h(h_imp[:, N]))
atenuacao_exp = (1.0 - Q_max_saida_exp / 12.0) * 100
atenuacao_imp = (1.0 - Q_max_saida_imp / 12.0) * 100

# Variação de lâmina d'água na jusante
h_max_exp = np.max(h_exp[:, N])
h_max_imp = np.max(h_imp[:, N])
delta_h_exp = h_max_exp - h0
delta_h_imp = h_max_imp - h0

print("=" * 60)
print(" ANÁLISE DE RESULTADOS")
print("=" * 60)
print(f"\n📊 Atenuação de Pico:")
print(f"  • Explícito: Q_max(saída) = {Q_max_saida_exp:.2f} m³/s → Atenuação = {atenuacao_exp:.1f}%")
print(f"  • Implícito: Q_max(saída) = {Q_max_saida_imp:.2f} m³/s → Atenuação = {atenuacao_imp:.1f}%")
print(f"\n📏 Variação de Lâmina d'Água na Jusante:")
print(f"  • Explícito: Δh = {delta_h_exp * 100:.1f} cm")
print(f"  • Implícito: Δh = {delta_h_imp * 100:.1f} cm")
print(f"\n⏱️ Desempenho Computacional:")
print(f"  • Explícito: {passos_exp} passos, {tempo_exp:.4f} s")
print(f"  • Implícito: {passos_imp} passos, {tempo_imp:.4f} s")

In [ ]:
# ============================================================
# TABELA COMPARATIVA
# ============================================================

dados_comparacao = {
    "Característica": [
        "Passo de tempo (Δt) utilizado",
        "Número total de passos de tempo",
        "Tempo de processamento (CPU)",
        "Restrição de Estabilidade (CFL)",
        "Complexidade de Implementação",
        "Custo Computacional por Passo",
        "Atenuação do Pico na Jusante",
        "Variação de lâmina (Δh) na Jusante",
        "Indicação de Uso"
    ],
    "Esquema Explícito": [
        f"{dt_exp:.0f} s (limitado pelo CFL)",
        f"{passos_exp} passos",
        f"{tempo_exp:.4f} s",
        "Condicionalmente estável (CFL ≤ 1)",
        "Baixa (cálculo direto, sem matrizes)",
        "Baixo (operações escalares/vetoriais)",
        f"~{atenuacao_exp:.1f}%",
        f"{delta_h_exp * 100:.1f} cm",
        "Malhas grossas, Δt pequenos, prototipagem rápida"
    ],
    "Esquema Implícito": [
        f"{dt_imp:.0f} s (pode ser muito maior)",
        f"{passos_imp} passos",
        f"{tempo_imp:.4f} s",
        "Incondicionalmente estável",
        "Alta (montagem e solução de sistema linear)",
        "Alto (solução de matriz tridiagonal)",
        f"~{atenuacao_imp:.1f}%",
        f"{delta_h_imp * 100:.1f} cm",
        "Simulações de longa duração, malhas refinadas"
    ]
}

df_comparacao = pd.DataFrame(dados_comparacao)

print("=" * 60)
print(" TABELA COMPARATIVA: ESQUEMAS DE DIFERENÇAS FINITAS")
print("=" * 60)
display(Markdown(df_comparacao.to_markdown(index=False)))

---

## 🔍 Seção 9: Análise de Sensibilidade Temporal

Verificamos a convergência da solução variando o passo de tempo. Para o esquema explícito, testamos $\Delta t = 15$, 30 e 60 s. Para o implícito, $\Delta t = 60$, 150 e 300 s.

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE TEMPORAL
# ============================================================

dt_testes_exp = [15.0, 30.0, 60.0]
dt_testes_imp = [60.0, 150.0, 300.0]

resultados_sens = []

print("Executando análise de sensibilidade temporal...")
print("=" * 70)

for dt in dt_testes_exp:
    cfl_teste = (ck_inicial * dt) / dx
    if cfl_teste > 1.0:
        print(f"  ⚠️ Explícito Δt={dt:.0f}s: CFL={cfl_teste:.2f} > 1 → INSTÁVEL, pulando.")
        continue
    t_s, h_s, p_s = simular_explicito(dt)
    Q_max_s = np.max(Q_from_h(h_s[:, N]))
    atenuacao_s = (1.0 - Q_max_s / 12.0) * 100
    resultados_sens.append({
        'Esquema': 'Explícito', 'Δt (s)': dt, 'CFL': f'{cfl_teste:.3f}',
        'Passos': p_s, 'Atenuação (%)': f'{atenuacao_s:.2f}'
    })
    print(f"  ✅ Explícito Δt={dt:.0f}s: CFL={cfl_teste:.3f}, Atenuação={atenuacao_s:.2f}%")

for dt in dt_testes_imp:
    cfl_teste = (ck_inicial * dt) / dx
    t_s, h_s, p_s = simular_implicito(dt)
    Q_max_s = np.max(Q_from_h(h_s[:, N]))
    atenuacao_s = (1.0 - Q_max_s / 12.0) * 100
    resultados_sens.append({
        'Esquema': 'Implícito', 'Δt (s)': dt, 'CFL': f'{cfl_teste:.3f}',
        'Passos': p_s, 'Atenuação (%)': f'{atenuacao_s:.2f}'
    })
    print(f"  ✅ Implícito Δt={dt:.0f}s: CFL={cfl_teste:.3f}, Atenuação={atenuacao_s:.2f}%")

print("=" * 70)
df_sens = pd.DataFrame(resultados_sens)
display(Markdown(df_sens.to_markdown(index=False)))

print("\n💡 Interpretação:")
print("  • A atenuação deve convergir para um valor estável à medida que Δt diminui")
print("  • O esquema implícito permite Δt maiores sem perda de estabilidade")
print("  • O esquema explícito é limitado pela condição CFL")

---

## 💡 Seção 10: Discussão, Limitações e Aplicações Práticas

### Resultados Principais

1. **Atenuação de pico:** A atenuação na jusante é da ordem de poucos por cento, típica de rios de planície com baixa declividade. A onda cinemática, por não incluir termos de pressão, subestima a atenuação real.

2. **Variação de lâmina d'água:** O aumento de profundidade na jusante durante o pico é da ordem de dezenas de centímetros, informação crítica para dimensionamento de margens e diques.

3. **Comparação entre esquemas:** Os dois esquemas produzem resultados muito próximos, validando a implementação. O esquema implícico permite passos de tempo 5× maiores, reduzindo o número total de passos.

4. **Custo computacional:** Embora o esquema implícito tenha custo por passo maior (solução de sistema linear), o número reduzido de passos compensa em simulações longas.

### Limitações do Modelo

- **Onda cinemática:** Despreza termos de inércia e gradiente de pressão. Para escoamentos com aceleração significativa (ressaltos hidráulicos, vertedouros), é necessário usar as equações completas (onda dinâmica).

- **Dispersão numérica:** O esquema upwind de primeira ordem introduz difusão artificial, alargando a frente da onda. Esquemas de ordem superior (Lax-Wendroff, TVD) reduzem esse efeito.

- **Geometria simplificada:** O modelo assume seção retangular constante. Rios naturais têm geometria variável, exigindo modelos 1D com seção transversal variável ou modelos 2D.

- **Linearização no esquema implícito:** A aproximação $Q^{n+1} \approx Q^n + (dQ/dh)^n(h^{n+1}-h^n)$ é de primeira ordem. Para maior precisão, pode-se usar iterações de Newton-Raphson.

### Aplicações Práticas

- **Sistemas de alerta:** O tempo de trânsito permite prever quando a cheia chegará a pontos críticos.
- **Dimensionamento de estruturas:** A variação de lâmina d'água define a altura mínima de margens e diques.
- **Gestão de reservatórios:** A atenuação de pico informa a capacidade de amortecimento necessária.

> **📖 Referência:** Gobbi (s.d.), Cap. 5 — Poluição Hídrica, Seção 5.1.

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 5](../notebooks/05_Hidrodinamica_Canais_Abertos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

Estudo de Caso 5.1  
Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*

</div>